Enable autoreload so edits to `src/*.py` are picked up automatically without restarting the kernel.

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Imports: standard library, torch/torchvision, and the project's shared modules (`src/*`).

In [ ]:

import json
import random

import torch
from torch import nn, optim
from torch.utils.data import random_split, DataLoader, Subset
from torchvision.datasets import OxfordIIITPet

from src.dataset import define_transformations
from src.experiment import save_model, save_results, save_config, prepare_results_dir
from src.metrics import plot_training_metrics
from src.training import training_loop


Pick the training device: CUDA -> MPS (Apple Silicon) -> CPU.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Load hyperparameters from `configs/base_config.json`.

In [ ]:
with open("../configs/base_config.json", encoding="utf-8") as file:
    config_json = json.load(file)

seed = int(config_json["seed"])

batch_size = int(config_json["data"]["batch_size"])
augmentation = config_json["data"]["augmentation"]

model = config_json["model"]["name"]
dropout = config_json["model"]["dropout"]

learning_rate = float(config_json["training"]["learning_rate"])
weight_decay = float(config_json["training"]["weight_decay"])
scheduler_config = config_json["training"]["scheduler"]
epochs = int(config_json["training"]["epochs"])

early_stopping_config = config_json["training"]["early_stopping"]
early_stopping_patience = int(early_stopping_config["patience"])
early_stopping_min_delta = float(early_stopping_config["min_delta"])

Fix the random seed for reproducibility (random split, weight initialization, etc.).

In [ ]:
random.seed(seed)
torch.manual_seed(seed)

Preprocessing constants: image size and ImageNet mean/std for normalization.

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

Build the train/val transform pipelines from the config (`define_transformations`).

In [ ]:
train_transform, val_transform = define_transformations(augmentation, IMAGENET_MEAN, IMAGENET_STD)

Load the Oxford-IIIT Pet splits. `train_dataset` and `val_dataset` are two SEPARATE OxfordIIITPet instances (not one dataset split via `random_split` alone) so train/val get different transforms - a `Subset` of a single dataset would inherit that dataset's transform for both, silently running the random train-time augmentation on validation images too.

In [29]:
train_dataset = OxfordIIITPet(
    root="../data",
    split="trainval",
    target_types="category",
    download=True,
    transform=train_transform
)
val_dataset = OxfordIIITPet(
    root="../data",
    split="trainval",
    target_types="category",
    download=True,
    transform=val_transform
)
test_dataset = OxfordIIITPet(
    root="../data",
    split="test",
    target_types="category",
    download=True,
    transform=val_transform
)

Split trainval into train/val (70/30) using shared indices, and build the DataLoaders.

In [ ]:
generator = torch.Generator().manual_seed(seed)
train_indices, val_indices = random_split(train_dataset, [0.7, 0.3], generator=generator)

train_loader = DataLoader(Subset(train_dataset, train_indices.indices), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(Subset(val_dataset, val_indices.indices), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

AlexNet-style CNN architecture from scratch: 5 conv layers with BatchNorm/ReLU/MaxPool plus a fully-connected classifier.

In [ ]:
class MyAlexNet(nn.Module):
    def __init__(self, num_classes, dropout):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 96, 11, stride=4, padding=1)
        self.batchNorm1 = nn.BatchNorm2d(96)
        self.pool1 = nn.MaxPool2d(3, 2)

        self.conv2 = nn.Conv2d(96, 256, 5, stride=1, padding=2)
        self.batchNorm2 = nn.BatchNorm2d(256)
        self.pool2 = nn.MaxPool2d(3, 2)

        self.conv3 = nn.Conv2d(256, 384, 3, stride=1, padding=1)
        self.batchNorm3 = nn.BatchNorm2d(384)
        self.conv4 = nn.Conv2d(384, 384, 3, stride=1, padding=1)
        self.batchNorm4 = nn.BatchNorm2d(384)
        self.conv5 = nn.Conv2d(384, 256, 3, stride=1, padding=1)
        self.batchNorm5 = nn.BatchNorm2d(256)

        self.pool3 = nn.MaxPool2d(3, 2)

        self.relu = nn.ReLU()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(6400, 4096),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.relu(self.batchNorm1(self.conv1(x)))
        x = self.pool1(x)

        x = self.relu(self.batchNorm2(self.conv2(x)))
        x = self.pool2(x)

        x = self.relu(self.batchNorm3(self.conv3(x)))
        x = self.relu(self.batchNorm4(self.conv4(x)))
        x = self.relu(self.batchNorm5(self.conv5(x)))

        x = self.pool3(x)

        x = self.classifier(x)

        return x

Number of classes (breeds) in the dataset - needed for the output layer size.

In [ ]:
num_classes = len(train_dataset.classes)
print(f"{num_classes} classes found.")

Sanity check: build a throwaway model instance and print its structure.

In [ ]:
verify_simple_cnn = MyAlexNet(num_classes=10, dropout=0.5)
print("Model Structure:\n")
print(verify_simple_cnn)

Sanity check the forward pass on a fake input tensor - confirms the output shape is `[64, num_classes]`.

In [ ]:
# Create a dummy input tensor (batch_size=64, channels=3, height=32, width=32)
dummy_input = torch.randn(64, 3, 224, 224)
print(f"\nInput tensor shape:  {dummy_input.shape}")
# Pass the dummy tensor through the model
output = verify_simple_cnn(dummy_input)
print(f"Output tensor shape: {output.shape}")

Build the real model with the actual number of classes and the config's dropout.

In [ ]:
model = MyAlexNet(num_classes=num_classes, dropout=dropout)

Loss, optimizer (Adam) and LR scheduler (ReduceLROnPlateau) from the config.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=scheduler_config["mode"],
    factor=float(scheduler_config["factor"]),
    patience=int(scheduler_config["patience"]),
    min_lr=float(scheduler_config["min_lr"]),
)

Allocate a new `experiments/<N>` directory for this run.

In [ ]:
results_dir, exp_number, exp_path = prepare_results_dir("../experiments")
print("Results Directory:", results_dir)
print("Experiment Number:", exp_number)
print("Experiment Path:", exp_path)

Save this run's config into its experiment directory (`config.json`).

In [ ]:
save_config(exp_path, config_json)

Run training (`training_loop` stops early on its own if early stopping triggers).

In [ ]:
trained_model, training_metrics = training_loop(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    num_epochs=epochs,
    device=device,
    scheduler=scheduler,
    early_stopping_patience=early_stopping_patience,
    early_stopping_min_delta=early_stopping_min_delta
)

Plot and save the train/val loss and accuracy curves (`results.png`).

In [ ]:
print("\n--- Training Plots ---\n")
plot_training_metrics(training_metrics, exp_path)

Append this run's results to `experiments/summary.csv` and `metrics.csv`.

In [ ]:
save_results(results_dir,
             exp_path,
             exp_number,
             learning_rate,
             training_metrics,
             model.__class__.__name__,
             len(training_metrics[0]),
             batch_size,
             optimizer.__class__.__name__,
             weight_decay,
             seed,
             str(augmentation),
             dropout,
             scheduler_config
             )

Save the trained model's weights (`model.pt`).

In [ ]:
save_model(exp_path, model)